In [73]:
import great_expectations as gx
import pandas as pd

from great_expectations.core.validation_definition import ValidationDefinition
from great_expectations.checkpoint import (
    Checkpoint,
    UpdateDataDocsAction,
)
from great_expectations.core.run_identifier import RunIdentifier
from great_expectations.data_context.types.resource_identifiers import ExpectationSuiteIdentifier
from great_expectations.core.expectation_suite import ExpectationSuite

from pathlib import Path 



In [67]:
context = gx.get_context(mode="file", project_root_dir=Path("./VoronovaNastya_Task4"))

/opt/app/yandexMlops/final_5/yandex_mlops/5-final/.venv/lib/python3.12/site-packages/great_expectations/data_context/store/_store_backend.py:88: DeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parsed_store_backend_id = store_backend_id_file_parser.parseString(


In [68]:
# Настраиваем Data Source 
source_folder = Path("./")
data_source_name = "my_filesystem_data_source"

if data_source_name in context._datasources:
    context.delete_datasource(data_source_name)
    print(f"Datasource '{data_source_name}' удален")

data_source = context.data_sources.add_pandas_filesystem(
    name=data_source_name, base_directory=source_folder
)

Datasource 'my_filesystem_data_source' удален


In [69]:
# Настраиваем Data Asset
asset_name = "dataset_csv_files"

file_csv_asset = data_source.add_csv_asset(name=asset_name)

In [70]:
# Настраиваем Batch Definition
batch_definition_name = "dataset1.csv"
batch_definition_path = "dataset1.csv"

try:
    batch_definition = file_csv_asset.add_batch_definition_path(
            name=batch_definition_name, path=batch_definition_path
        )
except:
    print(f"batch definition уже существует для {batch_definition_name}")
    batch_definition = file_csv_asset.get_batch_definition(batch_definition_name)

In [71]:
# Проверим, что пакет данных читается средствами Great Expectations.
batch = batch_definition.get_batch()
print(batch.head())

Calculating Metrics: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 248.88it/s]

   Unnamed: 0                track_id                 artists  \
0           0  5SuOikwiRyPMVoIQDJUgSV             Gen Hoshino   
1           1  4qPNDBW1i3p13qLCt0Ki3A            Ben Woodward   
2           2  1iJBSr7s7jYXzM8EGcbK5b  Ingrid Michaelson;ZAYN   
3           3  6lfxq3CG4xtTiEg7opyCyx            Kina Grannis   
4           4  5vjLSffimiIP26QG5WcN2K        Chord Overstreet   

                                          album_name  \
0                                             Comedy   
1                                   Ghost (Acoustic)   
2                                     To Begin Again   
3  Crazy Rich Asians (Original Motion Picture Sou...   
4                                            Hold On   

                   track_name  popularity  duration_ms  explicit  \
0                      Comedy          73       230666     False   
1            Ghost - Acoustic          55       149610     False   
2              To Begin Again          57       210826     False   


In [72]:
suite_name = "my_expectation_suite"
# Создаем набор пустых (пока) правил in-memory
suite = gx.ExpectationSuite(name=suite_name)
try:
    # Сохраняем набор пустых (пока) правил в проекте GX.
    suite = context.suites.add(suite)
except:
    # Загружаем уже существующий expectation suite_name
    print(f"Suite = {suite_name} уже был создан ранее!")
    suite = context.expectations_store.get(ExpectationSuiteIdentifier(name=suite_name))
    suite = ExpectationSuite(**suite)

Suite = my_expectation_suite уже был создан ранее!


In [74]:
df = pd.read_csv('dataset1.csv')
unique_track_genre = list(df["track_genre"].unique())

In [75]:


# ------------Колонки не должны содержать нулевые значения----------
columns_no_nulls = ["track_id", "artists", "album_name", "track_name", "popularity", "duration_ms", 
 "explicit", "danceability", "energy", "key", "loudness", "mode", "speechiness", 
 "acousticness", "instrumentalness", "liveness", "valence", "tempo", 
 "time_signature", "track_genre"]
for col in columns_no_nulls:
    suite.add_expectation(
      gx.expectations.ExpectColumnValuesToNotBeNull(column=col, mostly=0.95)
)

# -------------- Проверяем наличие требуемых колонок -------------------------
suite.add_expectation(
    gx.expectations.ExpectTableColumnsToMatchSet(
        column_set=["track_id", "artists", "album_name", "track_name", "popularity", "duration_ms", 
 "explicit", "danceability", "energy", "key", "loudness", "mode", "speechiness", 
 "acousticness", "instrumentalness", "liveness", "valence", "tempo", 
 "time_signature", "track_genre"],
        exact_match=False,  # все указанные колонки должны быть в таблице. Дополнительные не запрещены
        meta={"notes": "Проверка, что все необходимые колонки существуют"}
    )
)

suite.add_expectation(
    gx.expectations.ExpectColumnValueLengthsToBeBetween(
        column="track_id",
        min_value=22,
        max_value=22,
        meta={"notes": "длина строки track_id строго 22 символа"}
    )
)

suite.add_expectation(
    gx.expectations.ExpectColumnValueLengthsToBeBetween(
        column="artists",
        min_value=2,
        max_value=512,
        meta={"notes": "длина строки artists  от 2 до 512 символов"}
    )
)

suite.add_expectation(
    gx.expectations.ExpectColumnValueLengthsToBeBetween(
        column="album_name",
        min_value=2,
        max_value=512,
        meta={"notes": "длина строки album_name  от 2 до 512 символов"}
    )
)

suite.add_expectation(
    gx.expectations.ExpectColumnValueLengthsToBeBetween(
        column="track_name",
        min_value=2,
        max_value=512,
        meta={"notes": "длина строки track_name  от 2 до 512 символов"}
    )
)

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="popularity",
        min_value=0,
        max_value=100,
        meta={"notes": "popularity в диапазоне [0, 100]"}
    )
)

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="duration_ms",
        min_value=0,
        max_value=5237760,
        meta={"notes": "duration_ms в диапазоне [0, 5237760]"}
    )
)

suite.add_expectation(
    gx.expectations.ExpectColumnValueLengthsToBeBetween(
        column="danceability",
        min_value=0,
        max_value=1,
        meta={"notes": "danceability  в диапазоне [0, 1]"}
    )
)

suite.add_expectation(
    gx.expectations.ExpectColumnValueLengthsToBeBetween(
        column="energy",
        min_value=0,
        max_value=1,
        meta={"notes": "energy в диапазоне [0, 1]"}
    )
)

suite.add_expectation(
    gx.expectations.ExpectColumnValueLengthsToBeBetween(
        column="key",
        min_value=0,
        max_value=11,
        meta={"notes": "key в диапазоне [0, 11]"}
    )
)

suite.add_expectation(
    gx.expectations.ExpectColumnValueLengthsToBeBetween(
        column="loudness",
        min_value=-45,
        max_value=5,
        meta={"notes": "loudness в диапазоне [-45, 5]"}
    )
)

suite.add_expectation(
    gx.expectations.ExpectColumnValueLengthsToBeBetween(
        column="mode",
        min_value=0,
        max_value=1,
        meta={"notes": "mode в диапазоне [0, 1]"}
    )
)

suite.add_expectation(
    gx.expectations.ExpectColumnValueLengthsToBeBetween(
        column="speechiness",
        min_value=0,
        max_value=1,
        meta={"notes": "speechiness в диапазоне [0, 1]"}
    )
)

suite.add_expectation(
    gx.expectations.ExpectColumnValueLengthsToBeBetween(
        column="acousticness",
        min_value=0,
        max_value=1,
        meta={"notes": "acousticness в диапазоне [0, 1]"}
    )
)

suite.add_expectation(
    gx.expectations.ExpectColumnValueLengthsToBeBetween(
        column="instrumentalness",
        min_value=0,
        max_value=1,
        meta={"notes": "instrumentalness в диапазоне [0, 1]"}
    )
)

suite.add_expectation(
    gx.expectations.ExpectColumnValueLengthsToBeBetween(
        column="liveness",
        min_value=0,
        max_value=1,
        meta={"notes": "liveness в диапазоне [0, 1]"}
    )
)

suite.add_expectation(
    gx.expectations.ExpectColumnValueLengthsToBeBetween(
        column="valence",
        min_value=0,
        max_value=1,
        meta={"notes": "valence в диапазоне [0, 1]"}
    )
)

suite.add_expectation(
    gx.expectations.ExpectColumnValueLengthsToBeBetween(
        column="tempo",
        min_value=0,
        max_value=256,
        meta={"notes": "tempo в диапазоне [0, 256]"}
    )
)

suite.add_expectation(
    gx.expectations.ExpectColumnValueLengthsToBeBetween(
        column="time_signature",
        min_value=0,
        max_value=5,
        meta={"notes": "tempo в диапазоне [0, 5]"}
    )
)

suite.add_expectation(
    gx.expectations.ExpectColumnDistinctValuesToBeInSet(
        column="track_genre",
        value_set=unique_track_genre
    )
)

ExpectColumnDistinctValuesToBeInSet(id='85a73891-33e7-4270-872c-f3a54aaf880b', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=False, rendered_content=None, windows=None, batch_id=None, column='track_genre', row_condition=None, condition_parser=None, value_set=['acoustic', 'afrobeat', 'alt-rock', 'alternative', 'ambient', 'anime', 'black-metal', 'bluegrass', 'blues', 'brazil', 'breakbeat', 'british', 'cantopop', 'chicago-house', 'children', 'chill', 'classical', 'club', 'comedy', 'country', 'dance', 'dancehall', 'death-metal', 'deep-house', 'detroit-techno', 'disco', 'disney', 'drum-and-bass', 'dub', 'dubstep', 'edm', 'electro', 'electronic', 'emo', 'folk', 'forro', 'french', 'funk', 'garage', 'german', 'gospel', 'goth', 'grindcore', 'groove', 'grunge', 'guitar', 'happy', 'hard-rock', 'hardcore', 'hardstyle', 'heavy-metal', 'hip-hop', 'honky-tonk', 'house', 'idm', 'indian', 'indie-pop', 'indie', 'industrial', 'iranian', 'j-dance', 

In [76]:
...

validation_name = "validation_run_1"

validation_definition = ValidationDefinition(
    name=validation_name,
    data=batch_definition,
    suite=suite
)

In [77]:
results = validation_definition.run()

Calculating Metrics: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 210/210 [00:02<00:00, 76.53it/s]


In [78]:
# Сгенерировать Data Docs
context.build_data_docs()

# Открыть Data Docs (откроется общий обзор)
context.open_data_docs()